# Round 4: Linear Regression - linreg, ridge, lasso, and elasticNet

In [ ]:
## ADD SAVE PARAMETERS
save = False
save_name = "round4_linreg_lasso"

In [15]:
# import statements
import numpy as np
import pandas as pd
import json, os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import sys, os

In [16]:
# load processed data
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
Y_train = pd.read_csv("../data/processed/Y_train.csv")

In [17]:
# Linear regression is sensitive to feature scale
# RF and LightGBM are NOT — but linear models ARE
scaler  = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)      # ← use same scaler, don't refit!

In [18]:
models = {
    'LinearRegression' : LinearRegression(),
    'Ridge (L2)'       : Ridge(alpha=1.0),
    'Lasso (L1)'       : Lasso(alpha=0.001),
    'ElasticNet'       : ElasticNet(alpha=0.001, l1_ratio=0.5),
}

results = {}

for name, model in models.items():
    cv_scores = cross_val_score(
        model, X_train_scaled, Y_train,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    results[name] = {
        'mean' : -cv_scores.mean(),
        'std'  : cv_scores.std(),
        'scores': (-cv_scores).tolist()
    }
    print(f"{name:<25} RMSE: {-cv_scores.mean():.5f} ± {cv_scores.std():.5f}")

LinearRegression          RMSE: 0.14876 ± 0.02553
Ridge (L2)                RMSE: 0.14871 ± 0.02554
Lasso (L1)                RMSE: 0.14762 ± 0.02621
ElasticNet                RMSE: 0.14812 ± 0.02601


In [19]:
# save to output

# Ridge is almost always the best linear model for this kind of data
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, Y_train)

# Lasso beats it in this task
lasso = Lasso(alpha=0.001)
lasso.fit(X_train_scaled, Y_train)

preds = np.expm1(lasso.predict(X_test_scaled))

submission = pd.DataFrame({
    'Id'       : pd.read_csv('../data/raw/test.csv')['Id'],
    'SalePrice': preds
})

if save:
    submission.to_csv(f'../data/output/{save_name}.csv', index=False)
    print(submission.head())

     Id      SalePrice
0  1461  117678.040947
1  1462  152640.483509
2  1463  169406.341963
3  1464  195508.193562
4  1465  182353.481518


In [20]:
# get cv_scores

cv_scores = cross_val_score(
    lasso, X_train, Y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

print(f"CV RMSE scores : {-cv_scores}")
print(f"Mean CV RMSE   : {-cv_scores.mean():.4f}")
print(f"Std CV RMSE    : {cv_scores.std():.4f}")

CV RMSE scores : [0.12513229 0.16081167 0.13262556 0.12797326 0.1977301 ]
Mean CV RMSE   : 0.1489
Std CV RMSE    : 0.0275


In [21]:
# save

log_entry = {
    "model"       : save_name,
    "cv_rmse_mean": round(float(-cv_scores.mean()), 5),
    "cv_rmse_std" : round(float(cv_scores.std()), 5),
    "cv_scores"   : [round(float(-s), 5) for s in cv_scores],
    "params"      : {},          # if using optuna, else {}
    "submission"  : f"{save_name}.csv",
    "notes"       : ""
}

if save: 
    os.makedirs('../data/output', exist_ok=True)
    log_path = '../data/output/cv_scores.json'
    # Load existing log or start fresh
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            log = json.load(f)
    else:
        log = []

    log.append(log_entry)

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"Logged CV score: {log_entry['cv_rmse_mean']:.5f} ± {log_entry['cv_rmse_std']:.5f}")
else: 
    print(f"Not logged, change to save boolean to log: {log_entry['cv_rmse_mean']:.5f} ± {log_entry['cv_rmse_std']:.5f}")

Logged CV score: 0.14885 ± 0.02755
